In [ ]:
# ==============================================================================
# SETUP
# ==============================================================================
from helpers import IncrementalPipeline, TableConfig, setup_logger
from datetime import datetime

# Setup logging
logger = setup_logger("incremental_dimensions")

# Initialize pipeline
batch_id = datetime.now().strftime("%Y%m%d_%H%M%S")
pipeline = IncrementalPipeline(spark, dbutils, batch_id=batch_id)

logger.info("Incremental dimension load initialized")

In [ ]:
# ==============================================================================
# SILVER TRANSFORMATION FUNCTIONS (Imported from shared module)
# ==============================================================================
from helpers.silver_transforms import (
    transform_customer_full_pipeline,
    transform_staff_full_pipeline,
    transform_store_full_pipeline,
    transform_car_full_pipeline
)

logger.info("Silver transformation functions imported from shared module")

In [ ]:
# ==============================================================================
# TABLE CONFIGURATIONS (DRY - Single Source of Truth)
# ==============================================================================

dimension_configs = [
    TableConfig(
        table_name="customer",
        business_key="customer_id",
        surrogate_key="customer_key",
        watermark_column="last_update",
        scd_type=1,
        gold_table_name="dim_customer",
        silver_transform=transform_customer_full_pipeline
    ),
    TableConfig(
        table_name="staff",
        business_key="staff_id",
        surrogate_key="staff_key",
        watermark_column="last_update",
        scd_type=1,
        gold_table_name="dim_staff",
        silver_transform=transform_staff_full_pipeline
    ),
    TableConfig(
        table_name="store",
        business_key="store_id",
        surrogate_key="store_key",
        watermark_column="last_update",
        scd_type=2,  # SCD Type 2 for manager changes
        tracking_columns=["manager_staff_id", "manager_first_name", "manager_last_name"],
        gold_table_name="dim_store",
        silver_transform=transform_store_full_pipeline
    ),
    TableConfig(
        table_name="inventory",
        business_key="inventory_id",
        surrogate_key="car_key",
        watermark_column="last_update",
        scd_type=1,
        gold_table_name="dim_car",
        silver_transform=transform_car_full_pipeline
    ),
]

logger.info(f"Configured {len(dimension_configs)} dimension tables")

In [ ]:
# ==============================================================================
# EXECUTE INCREMENTAL LOAD (DRY - Single Function Call)
# ==============================================================================

# Load all dimensions incrementally
results = pipeline.load_tables(dimension_configs, force_full=False)

# Display results
import pandas as pd
results_df = pd.DataFrame(results)
display(results_df)

In [ ]:
# Dimension table row counts
print("Dimension Table Row Counts:")
print(f"dim_customer: {spark.table('wheelie.gold.dim_customer').count():,}")
print(f"dim_staff: {spark.table('wheelie.gold.dim_staff').count():,}")
print(f"dim_store: {spark.table('wheelie.gold.dim_store').count():,}")
print(f"dim_car: {spark.table('wheelie.gold.dim_car').count():,}")

# Check latest watermarks
print("\nLatest Watermarks:")
display(spark.table("wheelie.monitoring.watermarks"))